# 多项式回归：用线性模型拟合非线性关系
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：多项式回归.py | 核心功能：PolynomialFeatures 特征扩展、degree 对比、Pipeline 封装

## 概述

线性回归只能拟合直线，但现实中的关系往往是弯曲的。多项式回归的思路很巧妙：先用 PolynomialFeatures 把原始特征扩展为高阶项（x, x², x³...），然后对扩展后的特征做线性回归——"在更大的特征空间里做线性"等价于"在原始空间里做非线性"。

脚本在三次多项式数据上演示了不同 degree 的拟合效果，以及过拟合的直观表现。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
# --- 导入库 ---
from sklearn.metrics import mean_squared_error, r2_score

## 数学原理

### 1. 多项式特征扩展

**代码对应**：`PolynomialFeatures(degree=d, include_bias=False)` 将原始特征扩展为多项式特征。

对于单变量 $x$，$d$ 阶多项式扩展为：

$$\phi(x) = (x, x^2, x^3, \ldots, x^d)$$

对于 $p$ 维特征 $\mathbf{x} = (x_1, \ldots, x_p)$，$d$ 阶多项式扩展包含所有次数 $\leq d$ 的单项式。例如 $d=2$，$p=2$ 时：

$$\phi(x_1, x_2) = (x_1, x_2, x_1^2, x_1 x_2, x_2^2)$$

**特征数量**：$d$ 阶、$p$ 维的多项式扩展后的特征数为：

$$N(d, p) = \binom{p + d}{d} = \frac{(p + d)!}{p! \cdot d!}$$

| $d$ \ $p$ | 1 | 2 | 3 | 5 | 10 |
|-----------|---|---|---|---|---|
| 2 | 2 | 5 | 9 | 20 | 65 |
| 3 | 3 | 9 | 19 | 55 | 285 |
| 5 | 5 | 20 | 55 | 251 | 3002 |

当 $p$ 和 $d$ 增大时，特征数**指数增长**——这就是维度灾难。

### 2. 核技巧与多项式核

**代码对应**：多项式回归等价于在原始特征空间做线性回归，但也可以通过**核技巧**避免显式计算高维特征。

多项式核函数：

$$K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}^T\mathbf{z} + c)^d$$

**关键性质**：$K(\mathbf{x}, \mathbf{z}) = \phi(\mathbf{x})^T\phi(\mathbf{z})$，即核函数值等于在高维空间中的内积。

对于 $d=2$，$p=2$，$c=1$：

$$K(\mathbf{x}, \mathbf{z}) = (x_1 z_1 + x_2 z_2 + 1)^2 = 1 + 2x_1 z_1 + 2x_2 z_2 + x_1^2 z_1^2 + 2x_1 x_2 z_1 z_2 + x_2^2 z_2^2$$

这等价于在 6 维空间中的内积，但只需在原始 2 维空间计算。

### 3. 过拟合与正则化

**代码对应**：代码中 `for degree in [1, 2, 3, 4, 5, 10]` 展示了不同阶数的效果。

高阶多项式容易过拟合。训练 R² 随阶数增大而增大（甚至到 1.0），但测试 R² 先增后减——经典的偏差-方差权衡曲线。

**正则化多项式回归**（Ridge/Lasso + PolynomialFeatures）可以缓解过拟合：

$$J(\mathbf{w}) = \|\mathbf{y} - \mathbf{\Phi}\mathbf{w}\|_2^2 + \alpha\|\mathbf{w}\|_2^2$$

其中 $\mathbf{\Phi}$ 为多项式扩展后的设计矩阵。

### 4. 多项式回归与 Taylor 展开

任何光滑函数 $f(x)$ 在 $x_0$ 附近可以用 Taylor 展开近似：

$$f(x) = f(x_0) + f'(x_0)(x - x_0) + \frac{f''(x_0)}{2!}(x - x_0)^2 + \cdots + \frac{f^{(d)}(x_0)}{d!}(x - x_0)^d + R_d(x)$$

多项式回归本质上是用数据来学习这些系数（$f^{(k)}(x_0)/k!$），从而逼近任意光滑函数。$d$ 越大，逼近越精确，但需要更多数据来准确估计系数。

### 1. 生成非线性数据

运行 1. 生成非线性数据 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
X = np.sort(np.random.uniform(-3, 3, 100)).reshape(-1, 1)
y = 0.5 * X.ravel() ** 3 - 2 * X.ravel() ** 2 + 3 * X.ravel() + np.random.randn(100) * 2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. 不同阶数的多项式回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("=== 不同阶数的多项式回归 ===")
for degree in [1, 2, 3, 4, 5, 10]:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("lr", LinearRegression())
# --- 继续 ---
    ])
    model.fit(X_train, y_train)
    train_r2 = model.score(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    n_features = model.named_steps["poly"].n_output_features_
    # 交叉验证
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
    print(f"  degree={degree:>2}: 特征数={n_features:>3}, 训练R²={train_r2:.4f}, "
          f"测试R²={test_r2:.4f}, CV-R²={cv_scores.mean():.4f}±{cv_scores.std():.4f}")

### 3. 特征扩展效果

运行 3. 特征扩展效果 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== PolynomialFeatures 特征扩展 ===")
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_poly2 = poly2.fit_transform(X[:5])
print(f"原始特征 (degree=1): {X[:5].ravel()}")
print(f"扩展后 (degree=2): {poly2.get_feature_names_out(['x'])}")
# --- 输出结果 ---
print(f"值:\n{X_poly2[:3].round(4)}")

poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_poly3 = poly3.fit_transform(X[:3])
print(f"\n扩展后 (degree=3): {poly3.get_feature_names_out(['x'])}")

### 4. Pipeline 构建完整流程

运行 4. Pipeline 构建完整流程 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Pipeline: 缩放 + 多项式 + 回归 ===")
pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("lr", LinearRegression())
])
# --- 训练模型 ---
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(f"degree=3 测试 MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"degree=3 测试 R²: {r2_score(y_test, y_pred):.4f}")

### 5. 前 10 个预测对比

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 预测值 vs 真实值（前 10 个）===")
for i in range(10):
    print(f"  真实={y_test[i]:>8.2f}, 预测={y_pred[i]:>8.2f}")

print("\n=== 多项式回归要点 ===")
print("- 通过 PolynomialFeatures 扩展特征空间，将非线性问题转化为线性问题")
print("- degree 越高模型越复杂，容易过拟合（高阶项噪声放大）")
print("- 一般 degree=2 或 3 足够，degree>5 很少使用")
print("- 多项式回归的参数数量随维度指数增长（维度灾难）")
# --- 输出结果 ---
print("- 推荐用 Pipeline 封装，避免数据泄露")

## 关键代码解释

### PolynomialFeatures 扩展

```python
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)
```

对一维特征 x，degree=3 会生成 [x, x², x³]。对二维特征 [x₁, x₂]，degree=2 会生成 [x₁, x₂, x₁², x₁x₂, x₂²]。**特征数量随维度和阶数指数增长**——这是维度灾难的一个来源。

### degree 的过拟合曲线

```python
for degree in [1, 2, 3, 4, 5, 10]:
```

degree=1 欠拟合（直线），degree=3 恰好匹配数据生成过程，degree=10 严重过拟合（训练 R² 接近 1，测试 R² 可能变负）。

## 使用示例

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LinearRegression())
])
model.fit(X_train, y_train)
```

## 注意事项

1. **degree 选择**：通常 2 或 3 足够，>5 几乎必然过拟合
2. **维度爆炸**：d 维特征 degree=p 扩展后有 C(d+p, p) 个特征
3. **必须用 Pipeline**：单独 fit PolynomialFeatures 再 fit LinearRegression 会造成信息泄露
4. **正则化配合**：高阶多项式 + Ridge/Lasso 可以控制过拟合

## 总结与延伸

以上代码展示了 **多项式回归** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **核方法**：核技巧可以在无限维特征空间中做线性回归，等价于非参数多项式回归
- **样条回归**：用分段多项式（B-Spline）替代全局多项式，局部性更好
- **特征交叉**：在实际工程中，degree=2 的交叉特征 x₁·x₂ 往往比高阶项更有用
